# Reaction prediction

This notebook helps to understand how reaction outcomes can be predicted over one step. The dataset in use is the training set of the USPTO-MIT (409k; since sampling is needed anyway, npo need for the validation or test set) as obtained here: https://github.com/CC-SXF/DataSet-USPTO 

Sampling was introduced in order to limit the computation time.

1) Load dependencies

In [7]:

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import DataStructs, rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from tqdm import tqdm


2) Load USPTO-MIT data

In [8]:

inputs_df = pd.read_csv(
    "uspto_mit_inputs.txt",
    header=None,
    names=["inputs"]
)


targets_df = pd.read_csv(
    "uspto_mit_targets.txt",
    header=None,
    names=["targets"]
)


# Combine into a single DataFrame
df = pd.concat([inputs_df, targets_df], axis=1)


print("Number of reactions:", len(df))
df.head()

Number of reactions: 409035


,inputs,targets
0,C 1 C C O C 1 . C C ( C ) C [Mg+] . C O N ( C ...,C C ( C ) C C ( = O ) c 1 c c c ( O ) n c 1
1,C N . O . O = C ( O ) c 1 c c c ( Cl ) c ( [N+...,C N c 1 c c c ( C ( = O ) O ) c c 1 [N+] ( = O...
2,C C n 1 c c ( C ( = O ) O ) c ( = O ) c 2 c c ...,C C n 1 c c ( C ( = O ) O ) c ( = O ) c 2 c c ...
3,C C ( C ) = C ( Cl ) N ( C ) C . C O C C ( C )...,C O C C ( C ) O c 1 c c ( O c 2 c n c ( C ( = ...
4,Cl c 1 c c 2 c ( Cl ) n c ( - c 3 c c n c c 3 ...,Cl c 1 c c 2 c ( N C c 3 c c c ( Cl ) c ( Cl )...


In [9]:
df = df.sample(100000, replace=True, random_state=42)
print(len(df))
df.head()

100000


,inputs,targets
121958,C C # N . C C ( C ) N . C S C ( = N C # N ) N ...,C C ( C ) N C ( = N C # N ) N C 1 C C C c 2 c ...
146867,Cl C ( Cl ) ( Cl ) Cl . O C c 1 c c c c ( C C ...,Cl C c 1 c c c c ( C C 2 C C 2 ) c 1
131932,C C ( C ) O C ( = O ) Cl . Cl C Cl . O . [Na+]...,C C ( C ) O C ( = O ) N 1 C C 2 C C ( C N ( C ...
365838,C C ( C ) [Si] ( Cl ) ( C ( C ) C ) C ( C ) C ...,C C ( C ) [Si] ( O C C 1 O C ( n 2 c c c 3 c n...
259178,C 1 C O C C O 1 . C C ( N c 1 n c ( Cl ) c c (...,C C ( N c 1 n c ( N c 2 c n c c n 2 ) c c ( N ...


In [10]:
# Clean whitespaces
df["inputs"] = df["inputs"].str.replace(" ", "", regex=False)

df["targets"] = df["targets"].str.replace(" ", "", regex=False)
df.head()

,inputs,targets
121958,CC#N.CC(C)N.CSC(=NC#N)NC1CCCc2ccccc21,CC(C)NC(=NC#N)NC1CCCc2ccccc21
146867,ClC(Cl)(Cl)Cl.OCc1cccc(CC2CC2)c1.c1ccc(P(c2ccc...,ClCc1cccc(CC2CC2)c1
131932,CC(C)OC(=O)Cl.ClCCl.O.[Na+].[OH-].c1ccc(CN2CC3...,CC(C)OC(=O)N1CC2CC(CN(Cc3ccccc3)C2)C1
365838,CC(C)[Si](Cl)(C(C)C)C(C)C.CN(C)C=O.O.OCC1OC(n2...,CC(C)[Si](OCC1OC(n2ccc3cnccc32)CC1O)(C(C)C)C(C)C
259178,C1COCCO1.CC(Nc1nc(Cl)cc(N2CC(=O)NC(=O)C2)n1)c1...,CC(Nc1nc(Nc2cnccn2)cc(N2CC(=O)NC(=O)C2)n1)c1cc...


3) Reaction featurisation

Combine reactants and reagents, ignore conditions, use MorganFPs.

In [11]:
# helper function 
def smiles_to_fp(smiles, n_bits=2048, radius=2):
    fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    fp = fpgen.GetFingerprint(mol)
    # return fp
    fp_arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)
    return fp_arr



In [12]:
X = []
y = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    rxn_smiles = str(row["inputs"])
    fp = smiles_to_fp(rxn_smiles)
    
    if fp is None:
        continue
    
    X.append(fp)
    y.append(row["targets"])

X = np.array(X)
y = np.array(y)

print("Usable reactions:", len(X))


  0%|          | 0/100000 [00:00<?, ?it/s][08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom without neighbors
[08:47:51] WARNING: not removing hydrogen atom w

Usable reactions: 100000


4) Convert Products to classification labels

We treat the prediction as multiclass prediction. This assumes that there is **only one product** for one reaction - which is mostly not true for reality. 

In [ ]:
product_to_id = {p: i for i, p in enumerate(sorted(set(y)))} # these Id's are our classes (encoding)
id_to_product = {i: p for p, i in product_to_id.items()} # if we need to go back aka retrosynthesis (decoding)

y_ids = np.array([product_to_id[p] for p in y])

print("Unique products:", len(product_to_id))

Unique products: 86769


5) Train/test split and optional sampling

In [14]:
MAX_SAMPLES = 20000  # set None to use full dataset

if MAX_SAMPLES:
    idx = np.random.choice(len(X), MAX_SAMPLES, replace=False)
    X = X[idx]
    y_ids = y_ids[idx]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_ids, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 16000
Test size: 4000


6) Define and train ML model.

Using a probabilistic model (e.g. as here: RF) allows us to use effciently evaluation by top-k results and makes interpretation easy.

In [15]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_leaf=10,
    min_samples_split=10,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=25,
                       min_samples_leaf=10, min_samples_split=10,
                       n_estimators=300, n_jobs=-1, random_state=42)

7) Top-k accuracy determination

As more reaction paths and therefor multiple different products are possible, looking at one reaction outcome is not very useful.

Instead we are looking at the top-k results for the evaluation. 

In [16]:
def top_k_accuracy(y_true, probs, k):
    top_k = np.argsort(probs, axis=1)[:, -k:]
    return np.mean([
        y_true[i] in top_k[i]
        for i in range(len(y_true))
    ])

In [ ]:
probs = model.predict_proba(X_test)

for k in [5, 10, 20, 50]:
    acc = top_k_accuracy(y_test, probs, k)
    print(f"Top-{k} accuracy: {acc:.3f}")
# this tells us the procentage on how likely our true product is in the top 5 of predicted products

Top-5 accuracy: 0.000
Top-10 accuracy: 0.000
Top-20 accuracy: 0.001
Top-50 accuracy: 0.002


In [18]:
i = np.random.randint(len(X_test))

true_product = id_to_product[y_test[i]]
top_preds = np.argsort(probs[i])[-5:][::-1]

print("True product:")
print(true_product)

print("\nTop-5 predicted products:")
for pid in top_preds:
    print("-", id_to_product[pid])

True product:
CC(C)(C)OC(=O)N1CCCC(C(O)c2ccc(F)c(Cl)c2)C1

Top-5 predicted products:
- CC(C)OC(=O)CC(=O)C(C)C
- C=C(c1ccc(F)cc1)c1ccc2nc(-c3ccc(CN4CC(C(=O)OC)C4)cc3F)sc2n1
- CC(C)(C)S(=O)N=C1CCOCC1
- CC(=O)N(C)c1ccnc(N2CCN(C(C)C)CC2)c1
- CC(C)CNC(=O)C(C)NCC(O)C(Cc1cc(F)cc(F)c1)NC(=O)OC(C)(C)C



9) Discussion

- How would other tasks be done (e.g. yield prediction)?
- Why are the top-k accuracies super low?

    low information, data is very sparce -> problems to learn the context -> need better data or better model
    main limitation per class only one datapoint
- What could be done to improve this predictor?
    more experimental inforamtion
- What experimental information is missing?
    conditions, eq, 
- How would this model be used inside a synthesis planner?

